### Create Fact Table

Reading Silver Data 

In [0]:
df_silver = spark.sql("SELECT * FROM parquet.`abfss://silver@carsaledatalake.dfs.core.windows.net/CarSales`")

In [0]:
df_silver.display()

### Reading all DIMS

In [0]:
df_branch = spark.sql("select * from cars_catalog.gold.dim_branch")
df_dealer = spark.sql("select * from cars_catalog.gold.dim_dealer")
df_model = spark.sql("select * from cars_catalog.gold.dim_model")
df_date = spark.sql("select * from cars_catalog.gold.dim_date")


### ## Bringing keys to the fact tables

In [0]:
df_fact = df_silver.join(df_branch, df_silver['Branch_ID'] == df_branch['Branch_ID'],how='left')\
                  .join(df_date, df_silver['Date_ID'] == df_date['Date_ID'],how='left')\
                  .join(df_dealer, df_silver['Dealer_ID'] == df_dealer['Dealer_ID'],how='left')\
                  .join(df_model, df_silver['Model_ID'] == df_model['Model_ID'],how='left')\
                  .select(df_silver['Revenue'],df_silver['Units_Sold'],df_silver['RevenuePerUnit'],df_branch['dim_branch_key'],df_date['dim_date_key'],df_dealer['dim_dealer_key'],df_model['dim_model_key'])

In [0]:
df_fact.display()

In [0]:
from delta.tables import DeltaTable

In [0]:
if spark.catalog.tableExists('cars_catalog.gold.fact'):
    ## creating DeltaTable object
    delta_tbl = DeltaTable.forPath(spark, 'abfss://gold@carsaledatalake.dfs.core.windows.net/dim_fact')
    # now apply merge statement
    # alias are used to telling it is a our target table(it is more readable)
    delta_tbl.alias('trg').merge(df_fact.alias('src'), 'trg.dim_branch_key = src.dim_branch_key AND trg.dim_date_key = src.dim_date_key AND trg.dim_dealer_key = src.dim_dealer_key AND trg.dim_model_key = src.dim_model_key').whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
else:
    df_fact.write.format('delta').mode('overwrite').option('path', 'abfss://gold@carsaledatalake.dfs.core.windows.net/dim_fact').saveAsTable('cars_catalog.gold.factSales')    

In [0]:
%sql
select * from cars_catalog.gold.factSales